In [1]:
import os
import sys
import clr
import time
import threading
import webbrowser

import pandas as pd

from dash import Dash, dcc, html, Input, Output, State, ctx
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ==========================================================
# 1. VARIABLES: TAG Y NOMBRE DE COLUMNA
# ==========================================================

VARIABLES = {
    "FCCPI10463.PV": "Presion_Descarga",
    "FCCFI10561M.DACA.PV": "Flujo_RG_kg_h",
    "FCCPI10562.PV": "Presion_Hacia_RG",
    "FCCZI10561.PV": "Apertura_AntiChoque_pct",
    "FCCZI10461.PV": "Apertura_AntiSurge_pct",
    "FCCFO10626.PV": "IGV_pct",
    "FCCSI70129B.PV": "RPM",
    "FCCPI13162A.PV": "Presion_Vacio",
    "FCCDX11401B.DACA.PV": "SV_005",
    "FCCLY12103.OVRDSELA.OP": "SV_002",
    "FCCTY12114.OVRDSELA.OP": "SV_001",
    "FCCHY11108.OVRDSELA.OP": "SV_004",
    "FCCPDX12601B.DACA.PV": "SV_003",
}

TAGS = list(VARIABLES.keys())
NOMBRES_VARIABLES = list(VARIABLES.values())


# ==========================================================
# 2. CONFIGURACIÓN GENERAL
# ==========================================================

# Histórico disponible al iniciar el programa
INICIO_HISTORICO = "NOW-30D"
FIN_HISTORICO = "NOW"

# Histórico a resolución de 1 minuto
INTERVALO_HISTORICO_SEG = 60

# Actualización del dashboard
ACTUALIZACION_SEG = 5


# ==========================================================
# 3. CONFIGURACIÓN UNIFORMANCE PHD
# ==========================================================

RUTA = r"C:\Program Files\Honeywell\Uniformance\Excel Companion"

if RUTA not in sys.path:
    sys.path.append(RUTA)

if hasattr(os, "add_dll_directory"):
    dll_dir = os.add_dll_directory(RUTA)

clr.AddReference(os.path.join(RUTA, "phdapinet.dll"))

from Uniformance.PHD import (
    PHDServer,
    PHDHistorian,
    Tag,
    SERVERVERSION,
    SAMPLETYPE,
    REDUCTIONTYPE,
    REDUCTIONOFFSET,
)


# ==========================================================
# 4. CONECTAR AL SERVIDOR
# ==========================================================

print("=" * 80)
print("CONECTANDO AL PHD")
print("=" * 80)

server = PHDServer(
    "10.2.18.69",
    SERVERVERSION.RAPI200,
)

server.ConnectToServer()

print("✅ Conectado al PHD")
print("Servidor:", server.HostName)
print("Puerto:", server.Port)
print("API:", server.APIVersion)


# ==========================================================
# 5. LOCKS
# ==========================================================

# Evita que dos consultas entren simultáneamente al PHD
phd_lock = threading.Lock()

# Protege el DataFrame global
data_lock = threading.Lock()


# ==========================================================
# 6. CREAR HISTORIAN
# ==========================================================

def crear_historian(
    inicio,
    fin,
    sample_type,
    frecuencia_seg=None,
):

    historian = PHDHistorian()

    historian.DefaultServer = server

    historian.StartTime = inicio
    historian.EndTime = fin

    historian.Sampletype = sample_type

    if frecuencia_seg is not None:

        historian.SampleFrequency = int(frecuencia_seg)
        historian.UseSampleFrequency = True

    else:

        historian.UseSampleFrequency = False

    return historian


# ==========================================================
# 7. CONFIGURAR TAG
# ==========================================================

def crear_tag(nombre_tag):

    tag = Tag()

    tag.TagName = nombre_tag
    tag.Server = server

    tag.ReductionType = getattr(
        REDUCTIONTYPE,
        "None"
    )

    tag.ReductionFrequency = 0

    tag.ReductionOffset = REDUCTIONOFFSET.Before

    return tag


# ==========================================================
# 8. CONVERTIR TIMESTAMP PHD
# ==========================================================

def convertir_timestamp(timestamp):

    return pd.to_datetime(
        timestamp,
        unit="D",
        origin="1899-12-30",
    )


# ==========================================================
# 9. OBTENER HISTÓRICO DE UN TAG
# ==========================================================

def obtener_historico_tag(nombre_tag):

    historian = crear_historian(
        inicio=INICIO_HISTORICO,
        fin=FIN_HISTORICO,
        sample_type=SAMPLETYPE.Average,
        frecuencia_seg=INTERVALO_HISTORICO_SEG,
    )

    tag = crear_tag(nombre_tag)

    with phd_lock:
        datos = historian.FetchDataList(tag)

    timestamps = list(datos.Timestamps)
    valores = list(datos.DataValues)

    if not timestamps:
        return pd.DataFrame()

    df_tag = pd.DataFrame({
        "Timestamp_PHD": timestamps,
        nombre_tag: valores,
    })

    df_tag[nombre_tag] = pd.to_numeric(
        df_tag[nombre_tag],
        errors="coerce",
    )

    df_tag["FechaHora"] = convertir_timestamp(
        df_tag["Timestamp_PHD"]
    ).dt.round("min")

    df_tag = df_tag.drop(
        columns=["Timestamp_PHD"]
    )

    df_tag = (
        df_tag
        .groupby(
            "FechaHora",
            as_index=False
        )[nombre_tag]
        .mean()
    )

    return df_tag


# ==========================================================
# 10. CARGAR LOS 30 DÍAS INICIALES
# ==========================================================

def cargar_historico():

    print("\n" + "=" * 80)
    print("CARGANDO HISTÓRICO DE 30 DÍAS")
    print("=" * 80)

    dataframes = []

    for i, nombre_tag in enumerate(TAGS, start=1):

        try:

            print(
                f"[{i}/{len(TAGS)}] "
                f"Consultando {nombre_tag}..."
            )

            df_tag = obtener_historico_tag(
                nombre_tag
            )

            if not df_tag.empty:

                dataframes.append(df_tag)

                print(
                    f"   ✅ {len(df_tag):,} datos"
                )

            else:

                print("   ⚠️ Sin datos")

        except Exception as e:

            print(
                f"   ❌ Error: {e}"
            )

    if not dataframes:

        raise RuntimeError(
            "No fue posible obtener datos del PHD."
        )

    df_resultado = dataframes[0]

    for df_tag in dataframes[1:]:

        df_resultado = pd.merge(
            df_resultado,
            df_tag,
            on="FechaHora",
            how="outer",
        )

    df_resultado = (
        df_resultado
        .sort_values("FechaHora")
        .reset_index(drop=True)
    )

    df_resultado = df_resultado.rename(
        columns=VARIABLES
    )

    columnas = [
        "FechaHora",
        *NOMBRES_VARIABLES,
    ]

    df_resultado = df_resultado.reindex(
        columns=columnas
    )

    print("\n✅ Histórico cargado")
    print(
        "Registros:",
        f"{len(df_resultado):,}"
    )

    if not df_resultado.empty:

        print(
            "Desde:",
            df_resultado["FechaHora"].min()
        )

        print(
            "Hasta:",
            df_resultado["FechaHora"].max()
        )

    return df_resultado


# ==========================================================
# 11. CARGAR DATAFRAME GLOBAL
# ==========================================================

df_global = cargar_historico()

ultima_actualizacion = None


# ==========================================================
# 12. LEER ÚLTIMO VALOR DE UN TAG
# ==========================================================

def obtener_valor_actual(nombre_tag):

    # Consulta solo los últimos segundos.
    historian = crear_historian(
        inicio="NOW-15S",
        fin="NOW",
        sample_type=SAMPLETYPE.Snapshot,
        frecuencia_seg=5,
    )

    tag = crear_tag(nombre_tag)

    with phd_lock:
        datos = historian.FetchDataList(tag)

    valores = list(datos.DataValues)

    if not valores:
        return None

    serie = pd.to_numeric(
        pd.Series(valores),
        errors="coerce",
    ).dropna()

    if serie.empty:
        return None

    return serie.iloc[-1]


# ==========================================================
# 13. ACTUALIZAR VALORES EN VIVO
# ==========================================================

def actualizar_datos_live():

    global df_global
    global ultima_actualizacion

    fila = {}

    # Mismo timestamp para las 13 variables
    ahora = pd.Timestamp.now().floor("s")

    fila["FechaHora"] = ahora

    valores_validos = 0

    for nombre_tag, nombre_variable in VARIABLES.items():

        try:

            valor = obtener_valor_actual(
                nombre_tag
            )

            fila[nombre_variable] = valor

            if valor is not None:
                valores_validos += 1

        except Exception as e:

            fila[nombre_variable] = None

            print(
                f"⚠️ Error en {nombre_tag}: {e}"
            )

    if valores_validos == 0:

        print(
            "⚠️ No se obtuvieron valores nuevos."
        )

        return

    nueva_fila = pd.DataFrame(
        [fila]
    )

    with data_lock:

        df_global = pd.concat(
            [
                df_global,
                nueva_fila,
            ],
            ignore_index=True,
        )

        # Eliminar duplicados
        df_global = (
            df_global
            .drop_duplicates(
                subset="FechaHora",
                keep="last",
            )
            .sort_values("FechaHora")
            .reset_index(drop=True)
        )

        # Mantener solamente los últimos 30 días
        limite = (
            pd.Timestamp.now()
            - pd.Timedelta(days=30)
        )

        df_global = df_global[
            df_global["FechaHora"] >= limite
        ].reset_index(drop=True)

    ultima_actualizacion = ahora


# ==========================================================
# 14. FILTRAR DATOS SEGÚN PERÍODO
# ==========================================================

def obtener_datos_periodo(
    variables,
    segundos,
):

    if not variables:
        return pd.DataFrame()

    limite = (
        pd.Timestamp.now()
        - pd.Timedelta(seconds=segundos)
    )

    with data_lock:

        columnas = [
            "FechaHora",
            *variables,
        ]

        datos = df_global[
            df_global["FechaHora"] >= limite
        ][columnas].copy()

    return datos


# ==========================================================
# 15. REDUCIR PUNTOS PARA PERÍODOS LARGOS
# ==========================================================

def optimizar_para_grafica(
    datos,
    variables,
    segundos,
):

    if datos.empty:
        return datos

    # Para períodos cortos conservamos máxima resolución
    if segundos <= 3600:
        return datos

    # Hasta 6 horas
    elif segundos <= 6 * 3600:
        frecuencia = "1min"

    # Hasta 24 horas
    elif segundos <= 24 * 3600:
        frecuencia = "2min"

    # Hasta 7 días
    elif segundos <= 7 * 86400:
        frecuencia = "5min"

    # Hasta 15 días
    elif segundos <= 15 * 86400:
        frecuencia = "10min"

    # Hasta 30 días
    else:
        frecuencia = "15min"

    datos = datos.set_index(
        "FechaHora"
    )

    datos = (
        datos[variables]
        .resample(frecuencia)
        .mean()
        .reset_index()
    )

    return datos


# ==========================================================
# 16. CREAR GRÁFICA
# ==========================================================

def crear_grafica(
    datos,
    variables,
    modo,
):

    if not variables:

        fig = go.Figure()

        fig.update_layout(
            title="Seleccione al menos una variable",
            height=500,
        )

        return fig

    if datos.empty:

        fig = go.Figure()

        fig.update_layout(
            title="No hay datos disponibles",
            height=500,
        )

        return fig

    # ------------------------------------------------------
    # MODO 1: GRÁFICAS SEPARADAS EN UNA SOLA FIGURA
    # ------------------------------------------------------

    if modo == "separadas":

        numero = len(variables)

        fig = make_subplots(
            rows=numero,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=(
                0.03 if numero < 8 else 0.015
            ),
            subplot_titles=variables,
        )

        for i, variable in enumerate(
            variables,
            start=1,
        ):

            fig.add_trace(
                go.Scattergl(
                    x=datos["FechaHora"],
                    y=datos[variable],
                    mode="lines",
                    name=variable,
                    connectgaps=False,
                    hovertemplate=(
                        "%{x}<br>"
                        "%{y}<extra>"
                        + variable
                        + "</extra>"
                    ),
                ),
                row=i,
                col=1,
            )

            fig.update_yaxes(
                title_text=variable,
                row=i,
                col=1,
            )

        fig.update_layout(
            height=max(
                550,
                numero * 230,
            ),
            title=(
                "Monitoreo de variables PHD"
            ),
            showlegend=False,
            hovermode="x unified",
            margin=dict(
                l=70,
                r=30,
                t=80,
                b=60,
            ),
        )

        fig.update_xaxes(
            title_text="Fecha / Hora",
            row=numero,
            col=1,
        )

        return fig

    # ------------------------------------------------------
    # MODO 2: TODAS SUPERPUESTAS
    # ------------------------------------------------------

    fig = go.Figure()

    for variable in variables:

        fig.add_trace(
            go.Scattergl(
                x=datos["FechaHora"],
                y=datos[variable],
                mode="lines",
                name=variable,
                connectgaps=False,
            )
        )

    fig.update_layout(
        height=650,
        title="Monitoreo de variables PHD",
        xaxis_title="Fecha / Hora",
        yaxis_title="Valor",
        hovermode="x unified",
        legend=dict(
            orientation="h",
            y=-0.2,
        ),
        margin=dict(
            l=60,
            r=30,
            t=80,
            b=100,
        ),
    )

    return fig


# ==========================================================
# 17. CREAR APLICACIÓN DASH
# ==========================================================

app = Dash(__name__)

app.title = "Dashboard PHD"


# ==========================================================
# 18. DISEÑO DEL DASHBOARD
# ==========================================================

app.layout = html.Div(

    style={
        "fontFamily": "Arial",
        "maxWidth": "1600px",
        "margin": "auto",
        "padding": "20px",
    },

    children=[

        html.H1(
            "Dashboard PHD - Monitoreo de Variables"
        ),

        html.Div(
            "Actualización automática cada 5 segundos",
            style={
                "marginBottom": "20px",
                "fontSize": "16px",
            },
        ),

        # ==================================================
        # SELECTOR DE VARIABLES
        # ==================================================

        html.Div([

            html.Label(
                "Variables:",
                style={
                    "fontWeight": "bold"
                },
            ),

            dcc.Dropdown(
                id="selector-variables",

                options=[
                    {
                        "label": variable,
                        "value": variable,
                    }
                    for variable
                    in NOMBRES_VARIABLES
                ],

                # Variables iniciales
                value=[
                    "RPM",
                    "Presion_Vacio",
                    "Flujo_RG_kg_h",
                ],

                multi=True,

                placeholder=(
                    "Seleccione las variables..."
                ),
            ),

            html.Div(
                style={
                    "marginTop": "10px"
                },
                children=[

                    html.Button(
                        "Seleccionar las 13",
                        id="btn-todas",
                        n_clicks=0,
                        style={
                            "marginRight": "10px"
                        },
                    ),

                    html.Button(
                        "Limpiar selección",
                        id="btn-limpiar",
                        n_clicks=0,
                    ),

                ],
            ),

        ]),

        html.Br(),

        # ==================================================
        # CONTROLES
        # ==================================================

        html.Div(

            style={
                "display": "flex",
                "gap": "30px",
                "flexWrap": "wrap",
            },

            children=[

                # Período
                html.Div([

                    html.Label(
                        "Período mostrado:",
                        style={
                            "fontWeight": "bold"
                        },
                    ),

                    dcc.Dropdown(
                        id="selector-periodo",

                        options=[

                            {
                                "label": "15 minutos",
                                "value": 15 * 60,
                            },

                            {
                                "label": "30 minutos",
                                "value": 30 * 60,
                            },

                            {
                                "label": "1 hora",
                                "value": 3600,
                            },

                            {
                                "label": "3 horas",
                                "value": 3 * 3600,
                            },

                            {
                                "label": "6 horas",
                                "value": 6 * 3600,
                            },

                            {
                                "label": "12 horas",
                                "value": 12 * 3600,
                            },

                            {
                                "label": "24 horas",
                                "value": 24 * 3600,
                            },

                            {
                                "label": "7 días",
                                "value": 7 * 86400,
                            },

                            {
                                "label": "15 días",
                                "value": 15 * 86400,
                            },

                            {
                                "label": "30 días",
                                "value": 30 * 86400,
                            },
                        ],

                        value=3600,

                        clearable=False,

                        style={
                            "width": "220px"
                        },
                    ),

                ]),

                # Vista
                html.Div([

                    html.Label(
                        "Forma de visualización:",
                        style={
                            "fontWeight": "bold"
                        },
                    ),

                    dcc.RadioItems(
                        id="selector-modo",

                        options=[
                            {
                                "label":
                                " Gráficas separadas",
                                "value":
                                "separadas",
                            },

                            {
                                "label":
                                " Superpuestas",
                                "value":
                                "superpuestas",
                            },
                        ],

                        value="separadas",
                    ),

                ]),

                # Pausa
                html.Div([

                    html.Label(
                        "Actualización:",
                        style={
                            "fontWeight": "bold"
                        },
                    ),

                    html.Br(),

                    html.Button(
                        "⏸ Pausar",
                        id="btn-pausa",
                        n_clicks=0,
                    ),

                ]),

            ],
        ),

        html.Hr(),

        # ==================================================
        # ESTADO
        # ==================================================

        html.Div(
            id="estado-actualizacion",
            style={
                "fontWeight": "bold",
                "marginBottom": "10px",
            },
        ),

        # ==================================================
        # GRÁFICA
        # ==================================================

        dcc.Loading(
            type="circle",

            children=[

                dcc.Graph(
                    id="grafica-principal",
                    config={
                        "displaylogo": False,
                        "scrollZoom": True,
                    },
                ),

            ],
        ),

        # ==================================================
        # INTERVALO 5 SEGUNDOS
        # ==================================================

        dcc.Interval(
            id="intervalo-actualizacion",
            interval=ACTUALIZACION_SEG * 1000,
            n_intervals=0,
            disabled=False,
        ),

    ],
)


# ==========================================================
# 19. BOTONES TODAS / LIMPIAR
# ==========================================================

@app.callback(

    Output(
        "selector-variables",
        "value"
    ),

    Input(
        "btn-todas",
        "n_clicks"
    ),

    Input(
        "btn-limpiar",
        "n_clicks"
    ),

    prevent_initial_call=True,
)

def seleccionar_variables(
    n_todas,
    n_limpiar,
):

    trigger = ctx.triggered_id

    if trigger == "btn-todas":

        return NOMBRES_VARIABLES

    if trigger == "btn-limpiar":

        return []

    return []


# ==========================================================
# 20. PAUSAR / REANUDAR
# ==========================================================

@app.callback(

    Output(
        "intervalo-actualizacion",
        "disabled"
    ),

    Output(
        "btn-pausa",
        "children"
    ),

    Input(
        "btn-pausa",
        "n_clicks"
    ),

    State(
        "intervalo-actualizacion",
        "disabled"
    ),

    prevent_initial_call=True,
)

def pausar_actualizacion(
    n_clicks,
    esta_pausado,
):

    nuevo_estado = not esta_pausado

    if nuevo_estado:

        texto = "▶ Reanudar"

    else:

        texto = "⏸ Pausar"

    return nuevo_estado, texto


# ==========================================================
# 21. ACTUALIZACIÓN Y GRÁFICA
# ==========================================================

@app.callback(

    Output(
        "grafica-principal",
        "figure"
    ),

    Output(
        "estado-actualizacion",
        "children"
    ),

    Input(
        "intervalo-actualizacion",
        "n_intervals"
    ),

    Input(
        "selector-variables",
        "value"
    ),

    Input(
        "selector-periodo",
        "value"
    ),

    Input(
        "selector-modo",
        "value"
    ),
)

def actualizar_dashboard(
    n_intervalos,
    variables,
    periodo,
    modo,
):

    global ultima_actualizacion

    # Solo consultar PHD cuando el disparador
    # sea el intervalo de 5 segundos.
    if ctx.triggered_id == "intervalo-actualizacion":

        try:

            actualizar_datos_live()

        except Exception as e:

            print(
                "❌ Error actualización live:",
                e
            )

    if not variables:

        variables = []

    datos = obtener_datos_periodo(
        variables=variables,
        segundos=periodo,
    )

    datos = optimizar_para_grafica(
        datos=datos,
        variables=variables,
        segundos=periodo,
    )

    figura = crear_grafica(
        datos=datos,
        variables=variables,
        modo=modo,
    )

    if ultima_actualizacion is not None:

        texto_estado = (
            "🟢 PHD conectado | "
            "Última actualización: "
            + ultima_actualizacion.strftime(
                "%d/%m/%Y %H:%M:%S"
            )
            + f" | Registros almacenados: "
            f"{len(df_global):,}"
        )

    else:

        texto_estado = (
            "🟢 Histórico cargado | "
            "Esperando primera actualización "
            "en vivo..."
        )

    return figura, texto_estado


# ==========================================================
# 22. ABRIR NAVEGADOR
# ==========================================================

def abrir_navegador():

    webbrowser.open_new(
        "http://127.0.0.1:8050"
    )


# ==========================================================
# 23. EJECUTAR
# ==========================================================

if __name__ == "__main__":

    print("\n" + "=" * 80)
    print("INICIANDO DASHBOARD")
    print("=" * 80)

    print(
        "Dirección:",
        "http://127.0.0.1:8050"
    )

    threading.Timer(
        1.5,
        abrir_navegador
    ).start()

    app.run(
        host="127.0.0.1",
        port=8050,
        debug=False,
    )

CONECTANDO AL PHD
✅ Conectado al PHD
Servidor: 10.2.18.69
Puerto: 3150
API: RAPI200

CARGANDO HISTÓRICO DE 30 DÍAS
[1/13] Consultando FCCPI10463.PV...
   ✅ 43,201 datos
[2/13] Consultando FCCFI10561M.DACA.PV...
   ✅ 43,201 datos
[3/13] Consultando FCCPI10562.PV...
   ✅ 43,201 datos
[4/13] Consultando FCCZI10561.PV...
   ✅ 43,201 datos
[5/13] Consultando FCCZI10461.PV...
   ✅ 43,201 datos
[6/13] Consultando FCCFO10626.PV...
   ✅ 43,201 datos
[7/13] Consultando FCCSI70129B.PV...
   ✅ 43,201 datos
[8/13] Consultando FCCPI13162A.PV...
   ✅ 43,201 datos
[9/13] Consultando FCCDX11401B.DACA.PV...
   ✅ 43,201 datos
[10/13] Consultando FCCLY12103.OVRDSELA.OP...
   ✅ 43,201 datos
[11/13] Consultando FCCTY12114.OVRDSELA.OP...
   ✅ 43,201 datos
[12/13] Consultando FCCHY11108.OVRDSELA.OP...
   ✅ 43,201 datos
[13/13] Consultando FCCPDX12601B.DACA.PV...
   ✅ 43,201 datos

✅ Histórico cargado
Registros: 43,201
Desde: 2026-08-15 20:46:00
Hasta: 2026-09-14 20:46:00

INICIANDO DASHBOARD
Dirección: http:/